# 07 - Make one absolute move


Run cells from top to bottom in a fresh Python kernel. This notebook stands alone.
Install and launch as described in [the tutorial index](README.md).
Hardware access is disabled by default. Set your own eight-digit serial; no unit is
selected automatically. Close other notebooks' connections and other control software
before accessing the device. One connection owns the USB receive path.


**Physical motion:** inspect cable slack, attached optics, and the complete possible
rotation path first. Existing direction/backlash settings determine the route; a small
angle difference does not guarantee a short path. Home/move may enable a disabled motor.
The example does not automatically home, retry, or return to zero.

Calls block until completion. Interrupting the cell normally delivers `KeyboardInterrupt`,
which triggers a profiled-stop attempt on the owning connection. Kernel shutdown, force
termination, a lost USB link, or interrupt delivery failure cannot guarantee a stop.
Use your apparatus's physical safety procedure if control is lost. A controller-reported
stop is not independent proof of mechanical rest. Closing USB leaves motor holding state
unchanged.


In [ ]:
from dataclasses import asdict

from k10cr2 import K10CR2

SERIAL = ""  # Copy the serial from discovery and verify it on the mount's label.
CONNECT_TO_DEVICE = False  # Set True only when ready for USB access.


def configured_mount(**settings):
    # Construction validates the serial and timeouts but does not open USB.
    return K10CR2(SERIAL, **settings)

In [ ]:
from k10cr2 import MotionError


def report_motion_error(error):
    print("Original failure:", type(error.original).__name__, str(error.original))
    print("Stop cleanup:", asdict(error.cleanup))
    if not error.cleanup.confirmed_stopped:
        print("PHYSICAL STATE UNKNOWN: follow the apparatus safety procedure.")
    print("Close and explicitly reconnect; inspect state before deciding on another motion.")

## Preview the requested target without USB

Change the angle to a target suitable for your apparatus. The controller requires an
already-established home reference; this notebook never homes automatically.


In [ ]:
from k10cr2.protocol import COUNTS_PER_REVOLUTION
from k10cr2.settings import resolve_target

TARGET_DEG = 10.0
counts = resolve_target(TARGET_DEG, 0)
print(
    {
        "requested_deg": TARGET_DEG,
        "target_counts": counts,
        "quantized_deg": counts * 360 / COUNTS_PER_REVOLUTION,
    }
)

## Explicit move with a progress callback

Enter `MOVE ` followed by the selected serial in the confirmation. The callback receives
fresh `Status` and `Position` objects on the motion owner's thread. Keep it quick and do
not call any mount methods from it: the operation lock is held. A callback exception also
triggers stop cleanup. Queries/progress output add latency and are not real-time telemetry.

A move already at its target returns `no_op=True`. Otherwise final fresh position/status
must agree with completion. A short move can finish without a captured busy flag or
completion message; `completion_received` is separate from the target/status checks.


In [ ]:
CONFIRM_MOVE = ""  # Type "MOVE " followed by the actual serial after reviewing TARGET_DEG.


def show_progress(state, position):
    print(f"{position.degrees:.6f} deg | busy={state.busy} | homed={state.homed}")


if CONNECT_TO_DEVICE and SERIAL and CONFIRM_MOVE == f"MOVE {SERIAL}":
    with configured_mount() as mount:
        print("Before:", asdict(mount.status()), asdict(mount.position()))
        try:
            result = mount.move_absolute(TARGET_DEG, timeout=120, on_progress=show_progress)
            print("Result:", asdict(result))
        except MotionError as error:
            report_motion_error(error)
            raise
else:
    print("Move skipped; it requires connection opt-in and the exact MOVE confirmation.")

The mount remains at the resulting target; there is no automatic return. Clear the
confirmation after use. After an interrupted or failed move, use the recovery procedure
in [08 - Stop and recovery](08_stop_and_recovery.ipynb).
